# 01 · Train FreqBlender and the detector teacher
Stages: auxiliary ResNet34 → FPNet → EfficientNet-B4 detector.
Only real **training** frames are used; fake training images are generated on the fly.
Real/fake validation images select detector checkpoints. The test split is not used here.

This is a paper-based implementation with explicit adaptations: approximate spatial
self-blending, supplied PNG crops, FaceNet identity features and normalized losses.
The default short schedule is a starting experiment, not a reproduction of published AUC.
See the embedded `docs/METHOD.md` for every difference.


This PNG dataset has no supplied video/source grouping. All video-named metrics, validation scores, thresholds and bootstrap groups below operate on individual images. They must be reported as image-level results, not source-video results. Existing folder splits are preserved.


## Runtime and embedded project setup
For training, select **Runtime → Change runtime type → GPU**. This cell unpacks
the complete project source into `/kaggle/working/freqblender_project` and installs support
dependencies. It keeps Kaggle's matched PyTorch/torchvision CUDA installation.
No GitHub fork or separate project ZIP upload is needed.


## Kaggle input and writable outputs
Attach `delno002/split-ffpp` to every notebook. Its root must contain
train/real, train/fake, val/real, val/fake, test/real and test/fake PNG folders.
Existing splits are preserved. Images are read directly from the attached dataset.
Save notebook outputs before ending a session. For later stages, attach the previous
saved output and set PREVIOUS_RUNS to its `freqblender/runs` directory.


In [ ]:
import shutil
import json, sys, subprocess
from pathlib import Path
import torch



FFPP_ROOT = Path("/kaggle/input/split-ffpp")  # EDIT if Kaggle mounts a different path
DRIVE_PROJECT = Path("/kaggle/working/freqblender")  # legacy variable name, local Kaggle path
DATA_ROOT = DRIVE_PROJECT / "prepared"
RUN_ROOT = DRIVE_PROJECT / "runs"
MANIFEST = DATA_ROOT / "manifest.csv"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
PREVIOUS_RUNS = None  # e.g. Path("/kaggle/input/previous-output/freqblender/runs")
if PREVIOUS_RUNS is not None:
    # Copy missing files only, preserving newer checkpoints from this session.
    previous = Path(PREVIOUS_RUNS)
    if not previous.is_dir():
        raise FileNotFoundError(previous)
    for source in previous.rglob("*"):
        target = RUN_ROOT / source.relative_to(previous)
        if source.is_file() and not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, target)
from freqblender.data import build_image_manifest
MANIFEST = build_image_manifest(FFPP_ROOT, MANIFEST)
versions = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(DRIVE_PROJECT / "environment.txt").write_text(versions)
print("Manifest:", MANIFEST)
print("Using image-level samples; original video/source identities are unavailable.")


In [ ]:
assert torch.cuda.is_available(), "Choose a GPU runtime before training."
from freqblender.data import read_manifest
table = read_manifest(MANIFEST)
display(table.groupby(["split", "label"]).size().unstack(fill_value=0))
SIZE = 224          # 400 is supported; use SAME size in all stages/ablations
BATCH_SIZE = 4      # real-image batch; proposed stage processes 6x this per forward
WORKERS = 2
SEED = 42
MAX_STEPS = 0       # 0 = full epochs; positive values are execution tests only
AMP = False         # FP32 recommended for small secant response differences
def train_stage(stage, name, epochs, architecture="efficientnet_b4", extra=()):
    command = [sys.executable, "-m", "freqblender.train", "--stage", stage,
               "--manifest", str(MANIFEST), "--output", str(RUN_ROOT/name),
               "--architecture", architecture, "--size", str(SIZE),
               "--batch-size", str(BATCH_SIZE), "--workers", str(WORKERS),
               "--epochs", str(epochs), "--seed", str(SEED),
               "--max-steps", str(MAX_STEPS), "--resume"]
    if AMP:
        command += ["--amp"]
    subprocess.run(command + list(map(str, extra)), check=True)
    return RUN_ROOT/name/"best.pt"


In [ ]:
WARMUP_EPOCHS = 10
PARSER_EPOCHS = 30  # paper: 200 at 400px; budget/configure explicitly
TEACHER_EPOCHS = 30
WARMUP = RUN_ROOT / "warmup" / "best.pt"
PARSER = RUN_ROOT / "parser" / "best.pt"
TEACHER = RUN_ROOT / "teacher_freqblender" / "best.pt"


## 1. Train the authenticity auxiliary
This ResNet34 learns real versus self-blended faces. FPNet later freezes its weights
while preserving the gradient from its predictions to reconstructed images.


In [ ]:
WARMUP = train_stage("warmup", "warmup", WARMUP_EPOCHS, "resnet34")


## 2. Prepare the frozen face-identity encoder
Default: FaceNet InceptionResnetV1 pretrained on VGGFace2, a **substitute** for the
paper's MobileNet-ArcFace. Its weights download automatically on first use.
To use a compatible MobileNet-ArcFace TorchScript file, set `IDENTITY_CHECKPOINT`.
Contract: RGB [-1,1] at `IDENTITY_SIZE` -> [batch, embedding dimension]. The wrapper
normalizes embeddings. Do not use random identity weights or video IDs as identities.


In [ ]:
IDENTITY_CHECKPOINT = None  # e.g. DRIVE_PROJECT / "weights/mobilenet_arcface.ts"
IDENTITY_SIZE = 112
if IDENTITY_CHECKPOINT is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "facenet-pytorch==2.6.0"], check=True)
    from facenet_pytorch import InceptionResnetV1
from freqblender.models import IdentityEncoder
identity_check = IdentityEncoder(IDENTITY_CHECKPOINT, IDENTITY_SIZE).eval()
probe = torch.rand(2, 3, 160, 160, requires_grad=True)
embedding = identity_check(probe)
embedding[:, 0].sum().backward()
assert probe.grad is not None and torch.isfinite(probe.grad).all()
print(identity_check.kind, tuple(embedding.shape), "input gradients OK")
del identity_check, probe, embedding
(DRIVE_PROJECT / "environment.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))


In [ ]:
identity_args = [] if IDENTITY_CHECKPOINT is None else ["--identity-checkpoint", IDENTITY_CHECKPOINT,
                                                       "--identity-size", IDENTITY_SIZE]
PARSER = train_stage("parser", "parser", PARSER_EPOCHS,
                     extra=["--discriminator", WARMUP] + identity_args)


## Inspect frequency masks and synthesized images
Mask sums are softly regularized, not forced to one in the paper baseline.
Saturated/constant masks or badly distorted images indicate FPNet needs investigation
before spending compute on teacher/student training.


In [ ]:
import matplotlib.pyplot as plt
from freqblender.train import load_parser
from freqblender.data import make_loader
from freqblender.augment import spatial_self_blend
parser = load_parser(PARSER, "cuda")
batch = next(iter(make_loader(table[(table.split == "train") & (table.label == 0)], SIZE, 2, workers=0)))
real = batch["image"].cuda()
with torch.no_grad():
    fake = spatial_self_blend(real)
    _, masks = parser(fake)
    blended = parser.blend(real, fake)
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for axis, tensor, name in zip(axes[0], [real[0], fake[0], blended[0]], ["real", "spatial fake", "frequency blend"]):
    axis.imshow(tensor.cpu().permute(1, 2, 0).clamp(0, 1)); axis.set_title(name); axis.axis("off")
for axis, mask, name in zip(axes[1], masks[0], ["semantic", "structural", "noise"]):
    axis.imshow(mask.cpu(), vmin=0, vmax=1); axis.set_title(name); axis.axis("off")
plt.tight_layout()
print("Mean mask integrity error:", (masks.sum(1)-1).abs().mean().item())
del parser, real, fake, blended, masks
torch.cuda.empty_cache()


## 3. Train the FreqBlender teacher
FPNet is frozen. Each self-blended fake receives frequency blending with probability 0.2.
All detector inputs use the same RGB range and built-in ImageNet normalization.


In [ ]:
TEACHER = train_stage("teacher", "teacher_freqblender", TEACHER_EPOCHS,
                      extra=["--parser", PARSER])


## Optional but recommended: spatial-only baseline
This isolates the benefit of frequency augmentation within this implementation.
It uses the same B4 architecture, initialization seed, schedule and resolution.


In [ ]:
RUN_SPATIAL_BASELINE = True
if RUN_SPATIAL_BASELINE:
    train_stage("teacher", "teacher_spatial", TEACHER_EPOCHS)


In [ ]:
import pandas as pd
for name in ("warmup", "parser", "teacher_freqblender", "teacher_spatial"):
    history = RUN_ROOT / name / "history.json"
    if history.exists():
        display(pd.DataFrame(json.loads(history.read_text())).tail().assign(run=name))


Checkpoints are saved after every epoch. Rerun the same cells to resume; changing
experiment-defining settings requires a new run directory. Open **02_train_efficient_student.ipynb** next.
